Background Subtraction
======================

Goal
----

-   We will learn about background subtraction with OpenCV.
-   We will see: **cv.createBackgroundSubtractorMOG2()**, **cv.createBackgroundSubtractorKNN()**.

Theory
------

Background subtraction is a major preprocessing step in many vision-based applications. For
example, consider the case of a visitor counter where a static camera takes the number of
visitors entering or leaving the room, or a traffic camera extracting information about the
vehicles etc. In all these cases, first you need to extract the person or vehicles alone.
Technically, you need to extract the moving foreground from static background.

OpenCV provides two popular methods: **MOG2** (Mixture of Gaussians) and **KNN** (K-Nearest
Neighbors). Both are based on the same principle: build a model of the background and classify
each pixel as background or foreground.

Below is an interactive example using OpenCV's built-in sample video.</VSCode.Cell>


In [ ]:
import asyncio

import cv2 as cv
import ipywidgets as widgets
from ipycanvas import Canvas, hold_canvas
from IPython.display import display

# open video
video_path = "../../../assets/vtest.avi"
cap = cv.VideoCapture(video_path)
_ok, _first = cap.read()
if not _ok:
    raise RuntimeError(f"Cannot open video: {video_path}")
total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
cap.set(cv.CAP_PROP_POS_FRAMES, 0)

H, W = _first.shape[:2]
CANVAS_W = W * 2 + 10  # side by side with gap

# canvas — one for frame + mask side-by-side
canvas = Canvas(width=CANVAS_W, height=H)

# widgets
algo_dd = widgets.Dropdown(
    options=["MOG2", "KNN"], value="MOG2", description="Algorithm"
)
frame_sl = widgets.IntSlider(value=0, min=0, max=total_frames - 1, description="Frame")
play_btn = widgets.Button(description="▶ Play", button_style="success")

backSub = None
_playing = False
_play_task = None


def _init_backsub():
    global backSub
    if algo_dd.value == "MOG2":
        backSub = cv.createBackgroundSubtractorMOG2()
    else:
        backSub = cv.createBackgroundSubtractorKNN()


_init_backsub()


def _render(frame, frame_num):
    """Apply background subtraction and draw side-by-side to canvas."""
    fgMask = backSub.apply(frame)
    # frame number overlay
    cv.rectangle(frame, (10, 2), (100, 20), (255, 255, 255), -1)
    cv.putText(frame, str(frame_num), (15, 15), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0))
    # side-by-side: frame | mask
    fgMaskBGR = cv.cvtColor(fgMask, cv.COLOR_GRAY2BGR)
    side = cv.hconcat([frame, fgMaskBGR])
    with hold_canvas():
        canvas.put_image_data(cv.cvtColor(side, cv.COLOR_BGR2RGB), 0, 0)


def _process_frame(frame_num):
    """Seek to *frame_num*, read, and render (for slider / algo changes)."""
    cap.set(cv.CAP_PROP_POS_FRAMES, frame_num)
    ret, frame = cap.read()
    if ret:
        _render(frame, frame_num)


def _on_algo_change(*_):
    _stop_playback()
    _init_backsub()
    _process_frame(frame_sl.value)


def _on_frame_change(change):
    if _playing:
        return  # slider driven by playback coroutine — don't seek
    _process_frame(change["new"])


def _stop_playback():
    global _playing, _play_task
    if _play_task is not None:
        _play_task.cancel()
        _play_task = None
    _playing = False
    play_btn.description = "▶ Play"
    play_btn.button_style = "success"


async def _playback_loop(start_frame):
    """Read frames sequentially without blocking the Jupyter event loop."""
    global _playing, _play_task
    _playing = True
    f = start_frame
    cap.set(cv.CAP_PROP_POS_FRAMES, f)
    try:
        while f < total_frames:
            ret, frame = cap.read()
            if not ret:
                break
            _render(frame, f)
            frame_sl.value = f
            f += 1
            await asyncio.sleep(0.03)  # ~30 fps
    finally:
        if _play_task is asyncio.current_task():
            _playing = False
            _play_task = None
            play_btn.description = "▶ Play"
            play_btn.button_style = "success"


def _on_play(_):
    global _play_task
    if _play_task is not None:
        _stop_playback()
        return
    _play_task = asyncio.create_task(_playback_loop(frame_sl.value))
    play_btn.description = "⏸ Pause"
    play_btn.button_style = "warning"


algo_dd.observe(_on_algo_change, "value")
frame_sl.observe(_on_frame_change, "value")
play_btn.on_click(_on_play)

_process_frame(0)

display(
    widgets.VBox(
        [
            widgets.HBox([algo_dd, frame_sl, play_btn]),
            canvas,
        ]
    )
)